# Chunk 10 — Path Analysis & Counterfactual Ablation
This notebook analyzes the physical features learned by the GNN (such as current path length) and performs counterfactual ablations by flipping highly-attended vs. lowly-attended pixels to measure their impact on resonance.

---
## Cell 1 — Environment Setup


In [1]:
!pip install networkx scipy tqdm


---
## Cell 2 — Mount Drive and Setup Paths


In [2]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
REPO_ROOT = '/content/antenna-gnn'

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
sys.path.append(f'{REPO_ROOT}/src')

for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints', f'{DATA_ROOT}/figures', f'{DATA_ROOT}/splits']:
    os.makedirs(d, exist_ok=True)


Mounted at /content/drive
Cloning into '/content/antenna-gnn'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 141 (delta 70), reused 119 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 418.59 KiB | 3.67 MiB/s, done.
Resolving deltas: 100% (70/70), done.


---
## Cell 3 — Imports and Model Loading


In [4]:
import torch
import numpy as np
import pandas as pd
import json
import networkx as nx
import matplotlib.pyplot as plt
import zipfile
import shutil
from tqdm.auto import tqdm
from scipy.signal import find_peaks

!pip install torch_geometric
from model import AntennaGNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load standard model for ablation
model = AntennaGNN(hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201).to(device)
ckpt = torch.load(f'{DATA_ROOT}/checkpoints/best_model.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print("AntennaGNN loaded.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.3 MB/s eta 0:00:00
Device: cpu
AntennaGNN loaded.


---
## Cell 4 — Path Length Computation
Calculates the physical effective path length of the metal using a Breadth-First Search (BFS) over a NetworkX graph of the metal pixels.


In [ ]:
def compute_path_length_mm(patch_pattern, seed_mask, N):
    """Compute path-length metrics from a single BFS pass.
    Returns a dict with L_max_mm, L_p90_mm, L_median_mm, L_sqrt_area_mm.
    """
    # 1. Build a NetworkX Graph (undirected) with only metal pixels
    G = nx.Graph()
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                G.add_node(idx, pos=(i, j))
                # 4-connectivity edges (only check forward to avoid duplicates)
                if i + 1 < N and patch_pattern[i+1, j] == 1:
                    G.add_edge(idx, (i+1)*N + j)
                if j + 1 < N and patch_pattern[i, j+1] == 1:
                    G.add_edge(idx, i*N + j+1)

    # 2. Compute seed centroid
    coords = np.argwhere(seed_mask)
    if len(coords) == 0:
        return {'L_max_mm': 0, 'L_p90_mm': 0, 'L_median_mm': 0, 'L_sqrt_area_mm': 0}
    seed_r, seed_c = coords.mean(axis=0)

    # 3. Find the metal node closest to centroid
    min_dist = float('inf')
    feed_node = None
    for n, data in G.nodes(data=True):
        r, c = data['pos']
        dist = (r - seed_r)**2 + (c - seed_c)**2
        if dist < min_dist:
            min_dist = dist
            feed_node = n

    if feed_node is None:
        return {'L_max_mm': 0, 'L_p90_mm': 0, 'L_median_mm': 0, 'L_sqrt_area_mm': 0}

    # 4. Run BFS from feed node — single pass
    lengths = nx.single_source_shortest_path_length(G, feed_node)

    if not lengths:
        return {'L_max_mm': 0, 'L_p90_mm': 0, 'L_median_mm': 0, 'L_sqrt_area_mm': 0}

    depth_values = np.array(list(lengths.values()), dtype=float)
    pixel_size_mm = 32.375 / N

    return {
        'L_max_mm':       float(depth_values.max()) * pixel_size_mm,
        'L_p90_mm':       float(np.percentile(depth_values, 90)) * pixel_size_mm,
        'L_median_mm':    float(np.median(depth_values)) * pixel_size_mm,
        'L_sqrt_area_mm': float(np.sqrt(len(depth_values))) * pixel_size_mm,
    }


---
## Cell 5 — Compute Path Lengths for 25x25 Test Set
Iterates over all functioning test antennas, calculates their path length, and maps it to their actual resonant frequency.


In [ ]:
LOCAL_GRAPH_ROOT = '/content/local_graphs'
N = 25
dst_dir = f'{LOCAL_GRAPH_ROOT}/{N}x{N}'
done_marker = f'{dst_dir}/_CACHED.txt'
os.makedirs(dst_dir, exist_ok=True)

if not os.path.exists(done_marker):
    # The user noted the file is in Drive at the same path structure as the repo
    zip_path = f'{DATA_ROOT}/data/processed/processed_{N}x{N}.zip'

    if not os.path.exists(zip_path):
        # Check if it's just in the DATA_ROOT directly
        zip_path = f'{DATA_ROOT}/processed_{N}x{N}.zip'

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not locate {zip_path}. Please verify the file is in your Drive at antenna_gnn/data/processed/")

    print(f"Unzipping graphs from {zip_path} to {dst_dir}")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        pt_members = [m for m in zf.namelist() if m.endswith('.pt')]
        for member in tqdm(pt_members, desc=f'{N}x{N} unzip', unit='file'):
            basename = os.path.basename(member)
            with zf.open(member) as src, open(os.path.join(dst_dir, basename), 'wb') as dst:
                shutil.copyfileobj(src, dst)
    with open(done_marker, 'w') as fh:
        fh.write('DONE\n')

# Load artifacts
s11_mean = torch.tensor(np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')).to(device)
s11_std  = torch.tensor(np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')).to(device)
freq_axis = np.linspace(1.0, 4.0, 201)
seed_mask_25 = np.load(f'{DATA_ROOT}/artifacts/seed_mask_25.npy')

# Numpy versions for de-normalization
s11_mean_np = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy').flatten()
s11_std_np  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy').flatten()

def get_raw_s11(data):
    """De-normalize S11 from z-score normalized data.y back to dB scale."""
    if hasattr(data, 'y_raw') and data.y_raw is not None:
        return data.y_raw.squeeze(0).cpu().numpy().flatten()
    return (data.y.squeeze(0).cpu().numpy().flatten() * s11_std_np
            + s11_mean_np)

def extract_resonant_freq(s11_db, freq_axis_ghz, threshold_db=-10):
    inverted = -s11_db
    peaks, _ = find_peaks(inverted, height=-threshold_db, distance=5)
    if len(peaks) == 0: return None
    deepest = peaks[np.argmax(inverted[peaks])]
    return freq_axis_ghz[deepest]

with open(f'{DATA_ROOT}/splits/indices.json', 'r') as f:
    splits_all = json.load(f)
test_indices_25 = [e[1] for e in splits_all['test'] if e[0] == 25]

# --- Sanity check: first sample ---
first_sample_printed = False
agree_count = 0
agree_total = 0

results = []
for loop_i, idx in enumerate(tqdm(test_indices_25, desc="Computing path lengths")):
    pt_path = f'{dst_dir}/sample_{idx}.pt'
    if os.path.exists(pt_path):
        data = torch.load(pt_path, weights_only=False)
        raw_s11 = get_raw_s11(data)

        # One-time sanity print for first loaded sample
        if not first_sample_printed:
            norm_min = data.y.squeeze(0).cpu().numpy().min()
            raw_min = raw_s11.min()
            print(f"[Sanity] First sample idx={idx}: min(data.y)={norm_min:.4f}, min(raw_s11)={raw_min:.4f} dB")
            first_sample_printed = True

        # Agreement rate check over first 500 test samples
        if agree_total < 500:
            extracted_func = extract_resonant_freq(raw_s11, freq_axis) is not None
            if hasattr(data, 'is_functioning'):
                stored_func = bool(data.is_functioning)
                if extracted_func == stored_func:
                    agree_count += 1
            agree_total += 1

        res_freq = extract_resonant_freq(raw_s11, freq_axis)
        if res_freq is not None:
            pattern = data.x[:N*N, 0].view(N, N).cpu().numpy()
            metrics = compute_path_length_mm(pattern, seed_mask_25, N)
            results.append({
                'L_max_mm':       metrics['L_max_mm'],
                'L_p90_mm':       metrics['L_p90_mm'],
                'L_median_mm':    metrics['L_median_mm'],
                'L_sqrt_area_mm': metrics['L_sqrt_area_mm'],
                'res_freq_ghz':   res_freq
            })

df = pd.DataFrame(results)
csv_path = f'{DATA_ROOT}/artifacts/path_length_data_25x25.csv'
df.to_csv(csv_path, index=False)
print(f"Processed {len(df)} functioning antennas. Saved to {csv_path}")

if agree_total > 0 and agree_count > 0:
    rate = agree_count / agree_total * 100
    print(f"[Sanity] Agreement rate (is_functioning vs extract_resonant_freq) "
          f"over first {agree_total} samples: {agree_count}/{agree_total} = {rate:.1f}%")
elif agree_total > 0:
    print(f"[Sanity] No samples had is_functioning attribute; skipping agreement check.")


---
## Cell 6 — Scatter Plot: Path Length vs Resonant Frequency
We plot the true resonant frequency against the calculated geometric path length and overlay the theoretical formula $f = c / (2 L_{eff})$.


In [ ]:
from scipy.stats import pearsonr

metrics = ['L_max_mm', 'L_p90_mm', 'L_median_mm', 'L_sqrt_area_mm']
metric_labels = {
    'L_max_mm':       'L_max (mm)',
    'L_p90_mm':       'L_p90 (mm)',
    'L_median_mm':    'L_median (mm)',
    'L_sqrt_area_mm': r'$\sqrt{A}$ (mm)',
}

L_THRESHOLD = 37.5  # mm — below this, theory f=150/L predicts >4 GHz (outside band)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
summary_rows = []

for ax, metric in zip(axes.flat, metrics):
    L = df[metric].values
    f = df['res_freq_ghz'].values
    inv_f = 1.0 / f

    # Scatter
    ax.scatter(L, f, color='teal', alpha=0.3, s=10)

    # Theory overlay f=150/L clipped to panel's L range
    L_min, L_max = L.min(), L.max()
    L_theory = np.linspace(max(L_min, 1.0), L_max, 200)
    f_theory = 150.0 / L_theory
    # Clip to visible y-range
    mask_vis = (f_theory >= 0.5) & (f_theory <= 5.0)
    ax.plot(L_theory[mask_vis], f_theory[mask_vis], 'k--', alpha=0.7,
            label='Theory: f = 150/L')

    # R_all
    r_all, _ = pearsonr(L, inv_f)

    # R_inband (L >= 37.5 mm)
    inband = L >= L_THRESHOLD
    n_inband = inband.sum()
    if n_inband > 2:
        r_inband, _ = pearsonr(L[inband], inv_f[inband])
    else:
        r_inband = float('nan')

    # Shade L < threshold
    ax.axvspan(L_min - 1, L_THRESHOLD, color='lightgray', alpha=0.4, zorder=0)
    ax.text(L_THRESHOLD * 0.5, ax.get_ylim()[1] * 0.92 if ax.get_ylim()[1] > 0 else 3.8,
            'theory f > band edge', fontsize=7, ha='center', color='gray', style='italic')

    # Annotations
    txt = (f'R_all = {r_all:.3f}\n'
           f'R_inband = {r_inband:.3f}\n'
           f'n_inband = {n_inband}')
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, fontsize=9, va='top', ha='right',
            bbox=dict(facecolor='white', alpha=0.85, edgecolor='none'))

    ax.set_xlabel(metric_labels[metric])
    ax.set_ylabel('Resonant Frequency (GHz)')
    ax.set_title(metric_labels[metric])
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(True, linestyle='--', alpha=0.4)

    summary_rows.append({'metric': metric, 'R_all': r_all, 'R_inband': r_inband, 'n_inband': n_inband})

fig.suptitle('Resonant Frequency vs. Path-Length Metrics', fontsize=14, y=1.01)
plt.tight_layout()
save_path = f'{DATA_ROOT}/figures/path_length_vs_freq.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved 2\u00d72 scatter plot to {save_path}")

# Summary table
summary_df = pd.DataFrame(summary_rows)
print("\nCorrelation summary (L vs 1/f):")
print(summary_df.to_string(index=False))


---
## Cell 7 — Counterfactual Ablation Setup
We find the 10 highest-attention metal pixels (the "current path") and 10 low-attention metal pixels that are at a similar distance from the feed. These define our ablation sets.


In [ ]:
from torch_geometric.nn import global_mean_pool

# Redefine AttentionGNN from Chunk 9
class AttentionGNN(AntennaGNN):
    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch

        x = self.input_proj(x)
        edge_attr = self.edge_proj(edge_attr)

        attn_tuple = None
        for i, block in enumerate(self.blocks):
            for j, layer in enumerate(block):
                if i == len(self.blocks) - 1 and j == len(block) - 1:
                    conv_out, attn_tuple = layer.conv(x, edge_index, edge_attr=edge_attr, return_attention_weights=True)
                    out = layer.norm(conv_out)
                    x = layer.act(out + layer.residual_proj(x))
                else:
                    x = layer(x, edge_index, edge_attr)

        metal_mask = data.x[:, 0] > 0.5
        metal_x = x[metal_mask]
        metal_batch = batch[metal_mask]
        pooled = global_mean_pool(metal_x, metal_batch)

        virtual_mask = data.x[:, 3] == -1
        virtual_x = x[virtual_mask]

        combined = torch.cat([pooled, virtual_x], dim=-1)
        out = self.readout_proj(combined)
        out = self.output_mlp(out)

        return out, attn_tuple

attn_model = AttentionGNN(hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201).to(device)
attn_model.load_state_dict(ckpt['model_state'])
attn_model.eval()

def get_attention_map(data, model, N):
    data = data.to(device)
    with torch.no_grad():
        out, (attn_edge_index, attn_weights) = model(data)

    edge_scores = attn_weights.mean(dim=1).cpu()
    attn_edge_index = attn_edge_index.cpu()

    # GATv2Conv appends self-loop edges AFTER the original edges
    # (add_self_loops=True default). Slice to only original edges,
    # which align 1:1 with data.edge_index / data.edge_attr:
    n_original_edges = data.edge_index.size(1)
    original_edge_index = attn_edge_index[:, :n_original_edges]
    original_edge_scores = edge_scores[:n_original_edges]

    assert torch.equal(original_edge_index, data.edge_index.cpu()), (
        "attn_edge_index does not align with data.edge_index as expected."
    )

    edge_attr_cpu = data.edge_attr.cpu()
    metal_metal_mask = edge_attr_cpu[:, 0] == 0
    mm_edge_index = original_edge_index[:, metal_metal_mask]
    mm_edge_scores = original_edge_scores[metal_metal_mask]

    attention_grid = np.zeros(N * N)
    is_metal = data.x[:N*N, 0].cpu().numpy() > 0.5

    for i in range(N * N):
        if is_metal[i]:
            target_mask = (mm_edge_index[1] == i)
            if target_mask.any():
                attention_grid[i] = mm_edge_scores[target_mask].mean().item()

    attention_grid = attention_grid.reshape(N, N)
    min_val, max_val = attention_grid.min(), attention_grid.max()
    if max_val > min_val:
        attention_grid = (attention_grid - min_val) / (max_val - min_val)

    return attention_grid

def _find_feed_node_and_connected(pattern, seed_mask, N):
    """Build metal-only graph, find feed node, return (feed_node, connected_set, G)."""
    G = nx.Graph()
    for i in range(N):
        for j in range(N):
            if pattern[i, j] == 1:
                idx = i * N + j
                G.add_node(idx, pos=(i, j))
                if i + 1 < N and pattern[i+1, j] == 1:
                    G.add_edge(idx, (i+1)*N + j)
                if j + 1 < N and pattern[i, j+1] == 1:
                    G.add_edge(idx, i*N + j+1)
    coords = np.argwhere(seed_mask)
    if len(coords) == 0:
        return None, set(), G
    seed_r, seed_c = coords.mean(axis=0)
    min_dist = float('inf')
    feed_node = None
    for n, ndata in G.nodes(data=True):
        r, c = ndata['pos']
        d = (r - seed_r)**2 + (c - seed_c)**2
        if d < min_dist:
            min_dist = d
            feed_node = n
    if feed_node is None:
        return None, set(), G
    connected = set(nx.single_source_shortest_path_length(G, feed_node).keys())
    return feed_node, connected, G

# --- Single-loop candidate selection ---
np.random.seed(42)
shuffled_indices = np.random.permutation(test_indices_25)

ablation_tasks = []
coords = np.argwhere(seed_mask_25)
seed_r, seed_c = coords.mean(axis=0)

n_scanned = 0
n_skip_no_true_res = 0
n_skip_no_model_res = 0
TARGET_N = 50
MAX_SCAN = 600

for idx in tqdm(shuffled_indices, desc="Scanning candidates"):
    if len(ablation_tasks) >= TARGET_N or n_scanned >= MAX_SCAN:
        break
    n_scanned += 1

    pt_path = f'{LOCAL_GRAPH_ROOT}/25x25/sample_{idx}.pt'
    if not os.path.exists(pt_path):
        continue

    data = torch.load(pt_path, weights_only=False)

    # 1. Functioning check on RAW S11
    raw_s11 = get_raw_s11(data)
    if extract_resonant_freq(raw_s11, freq_axis) is None:
        n_skip_no_true_res += 1
        continue

    # 2. Attention map
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=device)
    attn_grid = get_attention_map(data, attn_model, N)
    pattern = data.x[:N*N, 0].view(N, N).cpu().numpy()

    # 3. Path / off-path pixel selection
    dist_grid = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            dist_grid[i, j] = np.sqrt((i - seed_r)**2 + (j - seed_c)**2)

    valid_metal = (pattern == 1) & (seed_mask_25 == 0)
    metal_indices = np.argwhere(valid_metal)
    metal_tuples = [tuple(x) for x in metal_indices]
    scores = [attn_grid[r, c] for r, c in metal_tuples]

    if len(metal_tuples) < 10:
        n_skip_no_true_res += 1  # not enough metal outside seed
        continue

    top_10_idx = np.argsort(scores)[-10:]
    path_pixels = [metal_tuples[i] for i in top_10_idx]

    path_dists = [dist_grid[r, c] for r, c in path_pixels]
    mean_dist = np.mean(path_dists)

    dist_mask = (dist_grid >= mean_dist * 0.8) & (dist_grid <= mean_dist * 1.2)
    valid_offpath = valid_metal & dist_mask
    offpath_indices = np.argwhere(valid_offpath)

    if len(offpath_indices) >= 10:
        offpath_tuples = [tuple(x) for x in offpath_indices]
        offpath_scores = [attn_grid[r, c] for r, c in offpath_tuples]
        bottom_10_idx = np.argsort(offpath_scores)[:10]
        offpath_pixels = [offpath_tuples[i] for i in bottom_10_idx]
    else:
        bottom_10_idx = np.argsort(scores)[:10]
        offpath_pixels = [metal_tuples[i] for i in bottom_10_idx]

    # 4. Baseline model prediction
    with torch.no_grad():
        out_orig = model(data)
    pred_s11_orig = (out_orig.cpu() * s11_std.cpu() + s11_mean.cpu()).numpy().flatten()
    orig_f = extract_resonant_freq(pred_s11_orig, freq_axis)

    if orig_f is None:
        n_skip_no_model_res += 1
        continue

    # 5. Feed node and pre-flip connected set for disconnect check
    feed_node, pre_flip_connected, _ = _find_feed_node_and_connected(pattern, seed_mask_25, N)

    # Assert selected pixels are never the feed node itself
    for r, c in path_pixels + offpath_pixels:
        assert r * N + c != feed_node, (
            f"Selected pixel ({r},{c}) is the feed node — seed block exclusion failed!")

    ablation_tasks.append({
        'antenna_id': int(idx),
        'pattern': pattern,
        'orig_f': orig_f,
        'orig_s11': pred_s11_orig,
        'path_pixels': path_pixels,
        'offpath_pixels': offpath_pixels,
        'feed_node': feed_node,
        'pre_flip_connected': pre_flip_connected,
    })

print(f"\nPrepared {len(ablation_tasks)} antennas for ablation.")
print(f"  Candidates scanned:              {n_scanned}")
print(f"  Skipped (no true resonance):     {n_skip_no_true_res}")
print(f"  Skipped (no model resonance):    {n_skip_no_model_res}")


---
## Cell 8 — Counterfactual Forward Passes
We systematically flip one pixel at a time (from metal to air), rebuild the PyG graph, and pass it through the model. We record how much the predicted resonant frequency shifts (`|Δ Resonant Frequency|`).


In [ ]:
from torch_geometric.data import Data

def build_pyg_graph(patch_pattern, seed_mask, N):
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])

    # Fix: The virtual node marker (index 3) must be -1.0 for the model to find it
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)

    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    batch = torch.zeros(N * N + 1, dtype=torch.long)

    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, batch=batch)

def _check_disconnect(pattern, flipped_r, flipped_c, feed_node, pre_flip_connected, N):
    """Check if flipping pixel (r,c) from metal to air disconnects any
    previously-feed-connected metal pixel."""
    flipped_idx = flipped_r * N + flipped_c

    # Build metal-only graph WITHOUT the flipped pixel
    G = nx.Graph()
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            if idx == flipped_idx:
                continue  # this pixel is now air
            if pattern[i, j] == 1:
                G.add_node(idx, pos=(i, j))
                if i + 1 < N and pattern[i+1, j] == 1 and (i+1)*N + j != flipped_idx:
                    G.add_edge(idx, (i+1)*N + j)
                if j + 1 < N and pattern[i, j+1] == 1 and i*N + j+1 != flipped_idx:
                    G.add_edge(idx, i*N + j+1)

    # BFS from feed_node in the modified graph
    if feed_node not in G:
        return True  # feed itself removed (shouldn't happen per our assert)
    post_flip_connected = set(nx.single_source_shortest_path_length(G, feed_node).keys())

    # Check: any node in pre_flip_connected (other than flipped) now unreachable?
    expected = pre_flip_connected - {flipped_idx}
    return not expected.issubset(post_flip_connected)

# Collect results in long-form
flip_records = []

for task in tqdm(ablation_tasks, desc="Ablating antennas"):
    antenna_id = task['antenna_id']
    orig_pat = task['pattern']
    orig_f = task['orig_f']
    orig_s11 = task['orig_s11']
    feed_node = task['feed_node']
    pre_flip_connected = task['pre_flip_connected']

    for condition, pixels in [('path', task['path_pixels']),
                               ('offpath', task['offpath_pixels'])]:
        for r, c in pixels:
            mod_pat = orig_pat.copy()
            mod_pat[r, c] = 0
            data = build_pyg_graph(mod_pat, seed_mask_25, 25).to(device)
            with torch.no_grad():
                out = model(data)
            pred_s11 = (out.cpu() * s11_std.cpu() + s11_mean.cpu()).numpy().flatten()
            mod_f = extract_resonant_freq(pred_s11, freq_axis)

            peak_delta = abs(mod_f - orig_f) if mod_f is not None else abs(0 - orig_f)
            curve_dist = float(np.linalg.norm(pred_s11 - orig_s11))
            disconnects = _check_disconnect(orig_pat, r, c, feed_node,
                                            pre_flip_connected, N)

            flip_records.append({
                'antenna_id': antenna_id,
                'condition': condition,
                'pixel_r': r,
                'pixel_c': c,
                'curve_dist': curve_dist,
                'peak_delta': peak_delta,
                'disconnects': disconnects,
            })

flip_df = pd.DataFrame(flip_records)
csv_path = f'{DATA_ROOT}/artifacts/ablation_flips_25x25.csv'
flip_df.to_csv(csv_path, index=False)

n_path = len(flip_df[flip_df['condition'] == 'path'])
n_offpath = len(flip_df[flip_df['condition'] == 'offpath'])
print(f"Evaluated {n_path} path pixel flips and {n_offpath} off-path pixel flips.")
print(f"Saved long-form results to {csv_path}")


---
## Cell 9 — Box Plot and Statistical Test
We visualize the ablation results using a paired per-antenna design.


---
## Cell 9 — Paired Ablation Statistics

**Rationale for the paired test.** Each antenna contributes 10 path and 10
off-path flips, so the \~1000 pooled flips are pseudo-replicated —
between-antenna variance in baseline fragility dominates the signal.
Pairing within antenna (median curve-distance per condition, one pair
per antenna) cancels antenna-level baseline fragility and yields a
proper n ≈ 50 Wilcoxon signed-rank test.


In [ ]:
from scipy.stats import mannwhitneyu, wilcoxon

# ── PRIMARY: Paired per-antenna Wilcoxon ──
per_antenna = flip_df.groupby(['antenna_id', 'condition'])['curve_dist'].median().unstack()
path_medians   = per_antenna['path'].values
offpath_medians = per_antenna['offpath'].values

w_stat, w_pval = wilcoxon(path_medians, offpath_medians, alternative='greater')
n_path_gt = np.sum(path_medians > offpath_medians)
n_total   = len(path_medians)
med_diff  = np.median(path_medians - offpath_medians)

print("=== PRIMARY: Paired per-antenna Wilcoxon signed-rank test ===")
print(f"  n antennas: {n_total}")
print(f"  Wilcoxon W = {w_stat:.1f}, p = {w_pval:.4e}")
print(f"  Antennas with path > offpath: {n_path_gt}/{n_total}")
print(f"  Median of per-antenna differences: {med_diff:.4f}")

# ── PRIMARY FIGURE: Paired slope plot ──
fig, ax = plt.subplots(figsize=(5, 7))
x_positions = [0, 1]

for i in range(n_total):
    color = 'crimson' if path_medians[i] > offpath_medians[i] else 'steelblue'
    ax.plot(x_positions, [path_medians[i], offpath_medians[i]],
            color=color, alpha=0.35, linewidth=0.8)

# Group medians
gm_path = np.median(path_medians)
gm_offpath = np.median(offpath_medians)
ax.plot(x_positions, [gm_path, gm_offpath], 'ko-', markersize=10, linewidth=2.5,
        zorder=10, label=f'Group median')

ax.set_xticks(x_positions)
ax.set_xticklabels(['Path\n(High Attention)', 'Off-path\n(Low Attention)'])
ax.set_ylabel('Median ||ΔS11|| per antenna')
ax.set_title('Counterfactual Ablation: Paired Per-Antenna Test')

annot = (f'Wilcoxon p = {w_pval:.2e}\n'
         f'Path > Off-path: {n_path_gt}/{n_total}\n'
         f'Median diff: {med_diff:.4f}')
ax.text(0.5, 0.97, annot, transform=ax.transAxes, fontsize=10,
        va='top', ha='center',
        bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray'))

ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/counterfactual_ablation_paired.png', dpi=300)
plt.show()

# ── SECONDARY: Pooled curve-distance box plot (pseudo-replicated, reference only) ──
path_dists_pooled = flip_df[flip_df['condition'] == 'path']['curve_dist'].values
offpath_dists_pooled = flip_df[flip_df['condition'] == 'offpath']['curve_dist'].values

stat_mw, pval_mw = mannwhitneyu(path_dists_pooled, offpath_dists_pooled,
                                alternative='greater')

fig, ax = plt.subplots(figsize=(6, 7))
box_data = [path_dists_pooled, offpath_dists_pooled]
labels = ['Current Path Pixels\n(High Attention)', 'Off-Path Pixels\n(Low Attention)']
bplot = ax.boxplot(box_data, tick_labels=labels, patch_artist=True, widths=0.5)
colors = ['#d1495b', '#5b8fa8']
for patch, color in zip(bplot['boxes'], colors):
    patch.set_facecolor(color)

ax.set_title('Pooled Curve Distance (pseudo-replicated, reference only)')
ax.set_ylabel('||ΔS11|| (L2 distance, 201-pt spectrum)')

# Place median labels ABOVE each box's upper whisker to avoid overlap
for i, (whisker_line, med_val) in enumerate(zip(bplot['whiskers'][1::2],
                                                 [np.median(path_dists_pooled),
                                                  np.median(offpath_dists_pooled)])):
    whisker_y = whisker_line.get_ydata().max()
    ax.text(i + 1, whisker_y * 1.05, f'Median: {med_val:.4f}',
            ha='center', va='bottom', fontsize=9)

ax.text(0.5, 0.03, f'Pooled Mann-Whitney U, p = {pval_mw:.4e}',
        transform=ax.transAxes, ha='center', fontsize=11)
plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/counterfactual_ablation_curve_distance.png', dpi=300)
plt.show()
print(f"\n[Reference] Pooled Mann-Whitney: median path={np.median(path_dists_pooled):.4f}, "
      f"offpath={np.median(offpath_dists_pooled):.4f}, p={pval_mw:.4e}")

# ── DISCONNECT DIAGNOSTICS ──
print("\n=== Disconnect diagnostics ===")
for cond in ['path', 'offpath']:
    sub = flip_df[flip_df['condition'] == cond]
    frac = sub['disconnects'].mean()
    print(f"  {cond}: {sub['disconnects'].sum()}/{len(sub)} flips disconnect ({frac:.1%})")

# Robustness: Wilcoxon excluding disconnecting flips
no_disc = flip_df[~flip_df['disconnects']]
if len(no_disc) > 0:
    per_ant_nd = no_disc.groupby(['antenna_id', 'condition'])['curve_dist'].median().unstack()
    # Only keep antennas that have both conditions after filtering
    per_ant_nd = per_ant_nd.dropna()
    if len(per_ant_nd) >= 5:
        w_nd, p_nd = wilcoxon(per_ant_nd['path'].values, per_ant_nd['offpath'].values,
                              alternative='greater')
        print(f"\n  [Robustness] Wilcoxon excl. disconnects: W={w_nd:.1f}, p={p_nd:.4e}, "
              f"n={len(per_ant_nd)} antennas")
    else:
        print(f"\n  [Robustness] Too few antennas ({len(per_ant_nd)}) after excluding disconnects.")

# ── SECONDARY: Peak-shift plot (kept as-is) ──
path_deltas_pooled = flip_df[flip_df['condition'] == 'path']['peak_delta'].values
offpath_deltas_pooled = flip_df[flip_df['condition'] == 'offpath']['peak_delta'].values

stat2, pval2 = mannwhitneyu(path_deltas_pooled, offpath_deltas_pooled,
                            alternative='greater')

fig, ax = plt.subplots(figsize=(6, 7))
bplot2 = ax.boxplot([path_deltas_pooled, offpath_deltas_pooled],
                    tick_labels=labels, patch_artist=True, widths=0.5)
for patch, color in zip(bplot2['boxes'], colors):
    patch.set_facecolor(color)
ax.set_title('Counterfactual Ablation: Discrete Peak Shift\n'
             '(secondary/reference metric)')
ax.set_ylabel('|Δ Resonant Frequency| (GHz)')
ax.text(0.5, 0.95, f'Mann-Whitney U, p = {pval2:.4e}',
        transform=ax.transAxes, ha='center', fontsize=11)
plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/counterfactual_ablation.png', dpi=300)
plt.show()
print(f"\n[Reference] Peak-shift median path: {np.median(path_deltas_pooled):.4f}, "
      f"offpath: {np.median(offpath_deltas_pooled):.4f}, p={pval2:.4e}")


---
## Regression Guards
Verify that key invariants are preserved after the edits above.


In [ ]:
import inspect

# --- Regression Guard 1: get_attention_map ---
src_attn = inspect.getsource(get_attention_map)
guard1a = 'attn_edge_index[:, :n_original_edges]' in src_attn
guard1b = '== 0' in src_attn and 'edge_attr' in src_attn
print(f"[Guard 1a] get_attention_map slices attn_edge_index[:, :n_original_edges]: "
      f"{'PASS' if guard1a else 'FAIL'}")
print(f"[Guard 1b] get_attention_map filters edge_attr on == 0: "
      f"{'PASS' if guard1b else 'FAIL'}")

# --- Regression Guard 2: orig_f / orig_s11 from model prediction ---
# The ablation task setup code is in the source of the cell that builds ablation_tasks.
# We check that the model(data) output is used, not ground truth.
# We verify by inspecting the task dicts: orig_s11 should NOT equal get_raw_s11
# (ground truth); it should be from model prediction.
# Additionally, check source code of the cell:
guard2_pass = True
for task in ablation_tasks[:3]:
    raw = get_raw_s11(torch.load(
        f'{LOCAL_GRAPH_ROOT}/25x25/sample_{task["antenna_id"]}.pt',
        weights_only=False))
    if np.allclose(task['orig_s11'], raw, atol=1e-4):
        guard2_pass = False
        break
print(f"[Guard 2]  orig_s11 comes from model prediction (not ground truth): "
      f"{'PASS' if guard2_pass else 'FAIL'}")

# --- Regression Guard 3: build_pyg_graph virtual node is_seed = -1 ---
src_build = inspect.getsource(build_pyg_graph)
guard3 = '-1.0' in src_build or '-1' in src_build
# More specifically check the virtual node line
guard3_specific = 'node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])' in src_build
print(f"[Guard 3]  build_pyg_graph sets virtual node is_seed = -1: "
      f"{'PASS' if guard3_specific else 'FAIL'}")

assert guard1a, "REGRESSION: get_attention_map missing edge slicing"
assert guard1b, "REGRESSION: get_attention_map missing edge_attr filter"
assert guard2_pass, "REGRESSION: orig_s11 appears to be ground truth"
assert guard3_specific, "REGRESSION: build_pyg_graph virtual node is_seed != -1"
print("\nAll regression guards PASSED.")
